In [105]:
import pandas as pd

df_train = pd.read_csv("train_data.csv")
df_test = pd.read_csv("test_data.csv")
composers = pd.read_csv("composers.csv")

df_train.head()

,id,pitch,duration
0,25,41,0.79
1,25,51,0.45
2,25,20,0.57
3,25,51,0.69
4,25,62,0.34


In [106]:
df_train.id.nunique()

400

In [107]:
composers = composers.sort_values(by='datapointID')
#composers

In [108]:
def get_feats(df_train):
    df_1 = pd.DataFrame(df_train.groupby("id")["pitch"].agg(["mean", "median", "min", "max", "std"]))
    df_1 = df_1.rename(columns={"mean": "mean_pitch", "median": "median_pitch", "min": "min_pitch", "max": "max_pitch", "std": "std_pitch"})
    df_2 = pd.DataFrame(df_train.groupby("id")["duration"].agg(["mean", "median", "min", "max", "std"]))
    df_2 = df_2.rename(columns={"mean": "mean_dur", "median": "median_dur", "min": "min_dur", "max": "max_dur", "std": "std_dur"})
    df_train = pd.concat([df_1, df_2], axis=1)
    df_train = df_train.reset_index()
    return df_train

df_train = get_feats(df_train)
df_train["composer"] = composers["answer"]

df_train.head()

,id,mean_pitch,median_pitch,min_pitch,max_pitch,std_pitch,mean_dur,median_dur,min_dur,max_dur,std_dur,composer
0,0,52.210059,52.0,10,91,18.363466,0.529349,0.520,0.10,1.18,0.256508,The Beatles
1,1,48.385224,47.0,8,87,17.916307,0.563588,0.560,0.09,1.18,0.266538,Psychedelic Corn Cruumpets
2,2,52.258427,53.0,13,94,19.046199,0.546629,0.515,0.12,1.02,0.263743,David Bowie
3,4,53.281690,54.0,15,87,18.030255,0.515915,0.470,0.08,1.04,0.269645,The Beatles
4,5,48.744292,48.0,12,90,18.853817,0.571279,0.610,0.10,1.12,0.252273,Pantera


In [109]:
df_test = get_feats(df_test)
df_test.head()

,id,mean_pitch,median_pitch,min_pitch,max_pitch,std_pitch,mean_dur,median_dur,min_dur,max_dur,std_dur
0,3,48.953678,49.0,7,88,17.981336,0.554251,0.57,0.10,1.15,0.260106
1,10,49.542522,51.0,7,85,18.882503,0.584751,0.60,0.10,1.16,0.261282
2,12,50.318898,49.0,14,89,17.881159,0.572205,0.58,0.11,1.12,0.254459
3,14,49.917073,50.0,11,90,17.920794,0.543366,0.55,0.10,1.05,0.253404
4,15,49.135802,49.0,8,92,18.966316,0.542675,0.52,0.10,1.10,0.264767


In [110]:
features = [col for col in df_train.columns if col != 'id' and col != 'composer']
target = 'composer'

X = df_train[features]
y = df_train[target]

X_final = df_test[features]

X.head()

,mean_pitch,median_pitch,min_pitch,max_pitch,std_pitch,mean_dur,median_dur,min_dur,max_dur,std_dur
0,52.210059,52.0,10,91,18.363466,0.529349,0.520,0.10,1.18,0.256508
1,48.385224,47.0,8,87,17.916307,0.563588,0.560,0.09,1.18,0.266538
2,52.258427,53.0,13,94,19.046199,0.546629,0.515,0.12,1.02,0.263743
3,53.281690,54.0,15,87,18.030255,0.515915,0.470,0.08,1.04,0.269645
4,48.744292,48.0,12,90,18.853817,0.571279,0.610,0.10,1.12,0.252273


In [111]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import cross_val_score

model = ExtraTreesClassifier()

cross_val_score(model, X, y, cv=3, scoring='f1_weighted').mean()

np.float64(0.2065487422297834)